In [8]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [23]:
# Путь к вашему GPKG файлу
input_file = r"X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\landuse.gpkg"  # Замените на ваш путь
layer_name = None  # Укажите имя слоя, если в файле несколько слоев

In [24]:
# Порог для удаления полей (95% пустых значений)
threshold_percent = 98

# Загрузка данных
print(f"Загрузка данных из {input_file}...")
gdf = gpd.read_file(input_file, layer=layer_name)
print(f"Загружено {len(gdf)} объектов с {len(gdf.columns)} полями")

# Сохраняем геометрию отдельно (её не будем анализировать)
geometry_col = gdf.geometry.name

# Анализ пустых значений по полям
print("\n" + "="*50)
print("Анализ пустых значений по полям:")
print("="*50)

# Список полей для удаления
fields_to_drop = []
for column in gdf.columns:
    # Пропускаем геометрию
    if column == geometry_col:
        continue
    
    # Подсчёт пустых значений (None, NaN, пустые строки)
    total_count = len(gdf)
    null_count = gdf[column].isna().sum()
    
    # Для строковых полей также считаем пустые строки
    if gdf[column].dtype == 'object':
        empty_string_count = (gdf[column] == '').sum()
        empty_string_count += (gdf[column].str.strip() == '').sum() if not gdf[column].isna().all() else 0
        null_count = max(null_count, empty_string_count)
    
    # Процент пустых значений
    null_percent = (null_count / total_count) * 100
    
    # Вывод информации
    print(f"{column:30} | Пустых: {null_count:6}/{total_count} ({null_percent:.1f}%)")
    
    # Добавляем в список для удаления если превышен порог
    if null_percent >= threshold_percent:
        fields_to_drop.append(column)
        print(f"  → Поле будет удалено")

Загрузка данных из X:\00_ЛабГрад_ПО\09_Сотрудники\Глазов Ю.А\izhevsk\landuse.gpkg...
Загружено 3074 объектов с 92 полями

Анализ пустых значений по полям:
full_id                        | Пустых:      0/3074 (0.0%)
osm_id                         | Пустых:      0/3074 (0.0%)
osm_type                       | Пустых:      0/3074 (0.0%)
landuse                        | Пустых:     13/3074 (0.4%)
full_name                      | Пустых:   3066/3074 (99.7%)
  → Поле будет удалено
addr:place                     | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
postal_code                    | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
is_in:district                 | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
is_in:city                     | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
office                         | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
contact:website                | Пустых:   3073/3074 (100.0%)
  → Поле будет удалено
contact:vk 

In [25]:
# Удаление полей
print("\n" + "="*50)
print(f"Удаление {len(fields_to_drop)} полей из {len(gdf.columns)-1} (исключая геометрию)")
print("="*50)

if fields_to_drop:
    print("\nУдаляемые поля:")
    for field in fields_to_drop:
        print(f"  - {field}")
    
    # Создаём копию без ненужных полей
    gdf_cleaned = gdf.drop(columns=fields_to_drop)
    
    # Сохранение результата
    output_file = input_file.replace('.gpkg', '_cleaned.gpkg')
    print(f"\nСохранение очищенного слоя в {output_file}...")
    gdf_cleaned.to_file(output_file, driver="GPKG")
    
    print(f"\n✓ Готово! Сохранено {len(gdf_cleaned)} объектов с {len(gdf_cleaned.columns)} полями")
    print(f"  Удалено полей: {len(fields_to_drop)}")
    print(f"  Осталось полей: {len(gdf_cleaned.columns)} (включая геометрию)")
else:
    print("\n✓ Нет полей для удаления (все поля заполнены более чем на 5%)")

# Дополнительная статистика
print("\n" + "="*50)
print("Итоговая статистика:")
print("="*50)
print(f"Исходный файл: {len(gdf.columns)} полей")
if fields_to_drop:
    print(f"Очищенный файл: {len(gdf_cleaned.columns)} полей")
    print(f"Размер исходного файла: {Path(input_file).stat().st_size / 1024 / 1024:.2f} MB")
    if Path(output_file).exists():
        print(f"Размер очищенного файла: {Path(output_file).stat().st_size / 1024 / 1024:.2f} MB")


Удаление 77 полей из 91 (исключая геометрию)

Удаляемые поля:
  - full_name
  - addr:place
  - postal_code
  - is_in:district
  - is_in:city
  - office
  - contact:website
  - contact:vk
  - contact:phone
  - contact:instagram
  - contact:facebook
  - contact:email
  - ref:temples.ru
  - plant:output:hot_water
  - fax
  - craft
  - landfill
  - building:levels
  - abandoned:farmland
  - resource
  - railway
  - public_transport
  - email
  - wood
  - population
  - tourism
  - resort
  - allotments
  - farmyard
  - parking
  - fee
  - depot
  - abandoned
  - name:ja
  - water
  - man_made
  - military
  - addr:country
  - building
  - natural
  - wikipedia
  - wikidata
  - short_name
  - phone
  - start_date
  - power
  - plant:source
  - plant:output:electricity
  - plant:method
  - frequency
  - amenity
  - opening_hours
  - industrial
  - alt_name
  - fuel:cng
  - construction:amenity
  - brand
  - operator
  - opening_date
  - construction
  - produce
  - leaf_type
  - leaf_cycle
